In [1]:
import xgboost as xgb
import pandas as pd
import src.train_utils as T
from sklearn.dummy import DummyRegressor

In [33]:
df = pd.read_csv('../datasets/exp_seasonality_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [34]:
train_df = T.add_spatial_lags(train_df, [3], ['delta_pm25_t'], pool='mean')
train_df = T.add_time_lags(train_df, [1, 2, 3], ['delta_pm25_t_mean_3x3'])

In [35]:
train_df = train_df.dropna()

In [36]:
train_df.columns

Index(['row', 'col', 'date', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t+1', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5', 'month', 'delta_pm25_t_mean_3x3',
       'delta_pm25_t_mean_3x3-lag1', 'delta_pm25_t_mean_3x3-lag2',
       'delta_pm25_t_mean_3x3-lag3'],
      dtype='object')

In [37]:
base = []

feature_experiments = [
    ('persistence', base),
]

model=DummyRegressor(strategy='constant', constant=0)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)
print(results.sort_values('weighted_rmse'))

Running experiment: persistence
Fold 0
RMSE: 1.8096570709844206
Fold 1
RMSE: 6.285164652466425
Fold 2
RMSE: 5.6869340421851575
Fold 3
RMSE: 9.69294640386119
Fold 4
RMSE: 14.102778143864887
Fold 5
RMSE: 2.2060055497204227
Fold 6
RMSE: 0.9905793334560528
Fold 7
RMSE: 1.0450960025186666
Fold 8
RMSE: 2.16115584308235
    experiment  n_features  unweighted_rmse  weighted_rmse
0  persistence           0         4.886702       6.964272


In [38]:
base = ['row', 'col', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5', 'month']


params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,
}

feature_experiments = [
    ('base', base),
    ('base + 3x3', base + ['delta_pm25_t_mean_3x3']),
    ('base + lag1', base + ['delta_pm25_t_mean_3x3-lag1']),
    ('base + lag2', base + ['delta_pm25_t_mean_3x3-lag2']),
    ('base + lag3', base + ['delta_pm25_t_mean_3x3-lag3']),
    ('base + 3x3 + lag1', base + ['delta_pm25_t_mean_3x3', 'delta_pm25_t_mean_3x3-lag1']),
    ('base + 3x3 + lag1 + lag2', base + ['delta_pm25_t_mean_3x3', 'delta_pm25_t_mean_3x3-lag1', 'delta_pm25_t_mean_3x3-lag2']),
    ('base + 3x3 + lag1 + lag2 + lag3', base + ['delta_pm25_t_mean_3x3', 'delta_pm25_t_mean_3x3-lag1', 'delta_pm25_t_mean_3x3-lag2', 'delta_pm25_t_mean_3x3-lag3']),

]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7246168352759566
Fold 1
RMSE: 5.389258572266513
Fold 2
RMSE: 5.547181958171492
Fold 3
RMSE: 8.860398358924018
Fold 4
RMSE: 12.66478703919406
Fold 5
RMSE: 2.1379379775795657
Fold 6
RMSE: 0.9838697439290336
Fold 7
RMSE: 1.041218174001573
Fold 8
RMSE: 2.0791692004865086
Running experiment: base + 3x3
Fold 0
RMSE: 1.7201397598752362
Fold 1
RMSE: 5.365219149166515
Fold 2
RMSE: 5.561997954545003
Fold 3
RMSE: 8.87196749969789
Fold 4
RMSE: 12.823842032117684
Fold 5
RMSE: 2.137645732505898
Fold 6
RMSE: 0.9758949553464343
Fold 7
RMSE: 1.0367010837657336
Fold 8
RMSE: 2.0720460066547357
Running experiment: base + lag1
Fold 0
RMSE: 1.7242614218472403
Fold 1
RMSE: 5.388262871280847
Fold 2
RMSE: 5.584263109681559
Fold 3
RMSE: 8.869419740720057
Fold 4
RMSE: 12.692636980337122
Fold 5
RMSE: 2.143761910986815
Fold 6
RMSE: 0.9802141598314086
Fold 7
RMSE: 1.0305056723990262
Fold 8
RMSE: 2.07053945159113
Running experiment: base + lag2
Fold 0
RMSE: 1.7243783050980763
